# 03b — One-dimensional Gaussian-process features

Fit each passband independently using a constant times Matern-3/2 kernel plus
white noise. Incorporate the scaled flux errors as observational variances.
Require five observations per band. The initial time scale is 20 days and its
optimization bounds are 10–1000 days. Use the posterior to summarize shape,
fit quality, residuals and signed integrated differential flux.

The GP is an interpolator rather than a physical transient model. Missing or
failed band fits use zero sentinels and a separate valid-band count.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))  

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.features import gp
from src.features.cache import write_cache, load_cache


## 1. Load the cleaned curves


In [ ]:
df = pd.read_csv(config.TRAIN_CLEAN_CSV)
print(f"{len(df):,} observations · {df.object_id.nunique():,} objects · bands {sorted(df.passband.unique())}")
df.head()


## 2. Inspect an individual GP fit


In [ ]:
oid = 252646  
light_curve = df[df.object_id == oid]

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for pb, letra in gp.PASSBAND_MAP.items():
    ax = axes.ravel()[pb]
    band_index = light_curve[light_curve.passband == pb]
    fit_result = gp.fit_object_gp(band_index, pb)
    if fit_result is None:
        ax.set_title(f"band {letra}: <{gp.MIN_OBS_BANDA} obs"); ax.set_axis_off(); continue
    estimator_model, t0 = fit_result['gp'], fit_result['t0']
    t = band_index.mjd.values - t0
    xx = np.linspace(t.min(), t.max(), 200).reshape(-1, 1)
    mu, sd = estimator_model.predict(xx, return_std=True)
    mu = mu * fit_result['flux_std'] + fit_result['flux_mean']      
    sd = sd * fit_result['flux_std']
    fluence = np.trapezoid(mu, xx.ravel())                  
    ipk = int(np.argmax(mu))                                
    ax.fill_between(xx.ravel(), 0, mu, color='C2', alpha=.18)  
    ax.errorbar(t, band_index.flux, yerr=band_index.flux_err, fmt='.', ms=4, alpha=.6, label='obs')
    ax.plot(xx, mu, '-', color='C3', lw=1.5, label='GP')
    ax.fill_between(xx.ravel(), mu - 2*sd, mu + 2*sd, color='C3', alpha=.15)
    ax.plot(xx.ravel()[ipk], mu[ipk], 'o', color='C3', ms=7, label='Peak')
    ax.axhline(0, color='k', lw=.5)
    ax.set_title(f"band {letra} · integrated flux={fluence:,.0f}")
axes.ravel()[0].legend(fontsize=8)
fig.suptitle(f"Object {oid} — GP fits by passband (area = integrated flux; point = peak)", y=1.02)
fig.supxlabel('Days since first observation'); fig.supylabel('flux')
plt.tight_layout()
fig.savefig(config.FIGURE_DIR / 'gp1d_ajuste_ejemplo.pdf', bbox_inches='tight')
plt.show()


### Descriptors of the same example

Extract temporal shape, kernel parameters, fit quality, residual summaries and
signed flux integrals from the posterior. The integrals are not bolometric
energies or calibrated physical fluences.


In [ ]:
feats = gp.object_gp_features(light_curve)
pd.Series(feats).to_frame('valor')


### Standardized residuals

Inspect residuals `(observation − posterior mean) / predictive standard deviation`
across passbands. Large residuals indicate disagreement with the interpolation
model and available support; they do not establish astrophysical novelty.


In [ ]:
resid = gp.object_gp_residuals(light_curve)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(resid, bins=40, color='C0', alpha=.8)
ax.axvline(0, color='k', lw=.8)
ax.set_title(f"Normalized GP residuals · object {oid}  (n={resid.size}, std={resid.std():.2f})")
ax.set_xlabel('(obs − μ) / σ'); ax.set_ylabel('Frequency')
plt.tight_layout(); plt.show()


## 3. Build the GP feature table


In [ ]:





RECOMPUTE_FEATURES = False
if RECOMPUTE_FEATURES:
    tabla_gp = gp.build_gp_table(df)
else:
    tabla_gp = load_cache(config.FEATURES_GP_CSV, 'gp1d')
    print('Validated existing feature cache; extraction was not repeated.')
print('tabla_gp:', tabla_gp.shape)
tabla_gp.head()


## 4. Check the schema and missing values


In [ ]:
assert tabla_gp.object_id.is_unique, 'object_id duplicado'
assert tabla_gp.isna().sum().sum() == 0, 'The table contains NaN values'
print(f"{tabla_gp.shape[0]:,} objects · {tabla_gp.shape[1]-1} features")
print('Bands fitted per object (gp_n_bands_ok):')
print(tabla_gp.gp_n_bandas_ok.value_counts().sort_index())


## 5. Descriptive separation from integrated flux


In [ ]:
metadata_table = pd.read_csv(config.METADATA_CLEAN_CSV)[['object_id', 'target']]
chk = tabla_gp.merge(metadata_table, on='object_id')
chk['log_fluence'] = np.sign(chk.gp_fluence_total) * np.log1p(chk.gp_fluence_total.abs())


display(chk.groupby('target')['log_fluence'].describe()[['count', 'mean', 'std']].sort_values('mean'))


sort_order = chk.groupby('target')['log_fluence'].median().sort_values().index
data = [chk.loc[chk.target == c, 'log_fluence'].values for c in sort_order]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.boxplot(data, showfliers=False)
ax.set_xticks(range(1, len(sort_order) + 1))
ax.set_xticklabels([str(c) for c in sort_order], rotation=45)
ax.set_xlabel('Target (PLAsTiCC class)'); ax.set_ylabel('Signed log integrated flux')
ax.set_title('Distribution of GP integrated flux by class')
plt.tight_layout(); plt.show()


## 6. Save the feature cache


In [ ]:
if RECOMPUTE_FEATURES:
    write_cache(tabla_gp, config.FEATURES_GP_CSV, 'gp1d', producer_name='notebooks/03b_gp1d_features.ipynb')
    print('saved ->', config.FEATURES_GP_CSV)
else:
    print('Retained existing feature CSV and manifest unchanged.')


In [ ]:
import pandas as pd, config
print(load_cache(config.FEATURES_GP_CSV, 'gp1d').object_id.nunique())
